<a href="https://colab.research.google.com/github/Parthpatil294/ML_6D_1BM23CS227/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
!pip install -q tabula-py
import tabula
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import time
import os

# Convert heart.pdf to heart.csv if the csv doesn't exist
if not os.path.exists('heart.csv') and os.path.exists('/content/heart.pdf'):
    print('Converting heart.pdf to heart.csv...')
    try:
        tables = tabula.read_pdf('/content/heart.pdf', pages='all', multiple_tables=True)
        if tables:
            df = pd.concat(tables)
            df.to_csv('heart.csv', index=False)
            print('Conversion successful.')
    except Exception as e:
        print(f'Error during conversion: {e}')

try:
    df = pd.read_csv('heart.csv')
    print('Dataset loaded successfully.')
    display(df.head())
except FileNotFoundError:
    print('heart.csv still not found. Please ensure the file is available.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 38.9 MB/s eta 0:00:00


Converting heart.pdf to heart.csv...
Conversion successful.
Dataset loaded successfully.


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,...,188,139,50,254,159,43,122,64,335,158
0,40.0,M,ATA,140.0,289.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,49.0,F,NAP,160.0,180.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,37.0,M,ATA,130.0,283.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,48.0,F,ASY,138.0,214.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,54.0,M,NAP,150.0,195.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# Robust Preprocessing for PDF-extracted data
# Drop columns that are entirely NaN
df_encoded = df.dropna(axis=1, how='all').copy()

# Identify target column
target_col = 'HeartDisease' if 'HeartDisease' in df_encoded.columns else df_encoded.columns[-1]

# Fill missing numeric values with median and categorical with mode instead of dropping all
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'object':
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].mode()[0] if not df_encoded[col].mode().empty else 'Unknown')
        if len(df_encoded[col].unique()) <= 2:
            df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col].astype(str))
        else:
            df_encoded = pd.get_dummies(df_encoded, columns=[col], drop_first=True)
    else:
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].median())

# Final check to ensure we have numeric data
df_encoded = df_encoded.apply(pd.to_numeric, errors='coerce').fillna(0)

if len(df_encoded) > 0:
    X = df_encoded.drop(target_col, axis=1)
    y = df_encoded[target_col]

    # Split and Scale
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print(f'Preprocessing complete. Samples: {len(df_encoded)}, Features: {X.shape[1]}')
else:
    print('Error: Dataframe is empty after cleaning. Please check PDF extraction.')

Preprocessing complete. Samples: 1798, Features: 102


In [11]:
def evaluate_models(X_tr, X_te, y_tr, y_te, label):
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100),
        'SVM': SVC(probability=True)
    }

    results = {}
    print(f'\n--- {label} ---')
    for name, model in models.items():
        start = time.time()
        model.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, model.predict(X_te))
        duration = time.time() - start
        results[name] = acc
        print(f'{name:20} | Accuracy: {acc:.4f} | Time: {duration:.4f}s')
    return results

# Evaluate base models
base_results = evaluate_models(X_train_scaled, X_test_scaled, y_train, y_test, 'Standard Scaled Results')


--- Standard Scaled Results ---
Logistic Regression  | Accuracy: 0.9972 | Time: 0.0347s
Random Forest        | Accuracy: 0.9972 | Time: 0.1898s
SVM                  | Accuracy: 0.9972 | Time: 0.1279s


In [12]:
# PCA Implementation
pca = PCA(n_components=0.95) # Retain 95% of variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f'Original dimensions: {X_train_scaled.shape[1]}')
print(f'Reduced dimensions: {X_train_pca.shape[1]}')

# Evaluate PCA models
pca_results = evaluate_models(X_train_pca, X_test_pca, y_train, y_test, 'PCA Model Results')

Original dimensions: 102
Reduced dimensions: 87

--- PCA Model Results ---
Logistic Regression  | Accuracy: 0.9972 | Time: 0.0624s
Random Forest        | Accuracy: 0.9944 | Time: 0.5094s
SVM                  | Accuracy: 0.9944 | Time: 0.1440s
